In [1]:
"""Python.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1u1IZbKkD9Y9mHXvV6czUI6W7ILr-sh6-
"""

'Python.ipynb\n\nAutomatically generated by Colab.\n\nOriginal file is located at\n    https://colab.research.google.com/drive/1u1IZbKkD9Y9mHXvV6czUI6W7ILr-sh6-\n'

In [3]:
!pip install earthengine-api

  Using cached httplib2-0.31.2-py3-none-any.whl.metadata (2.2 kB)
  Using cached uritemplate-4.2.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 478.0/478.0 kB 2.7 MB/s eta 0:00:002.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 14.9 MB/s eta 0:00:00m eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7 kB 15.1 MB/s eta 0:00:00
Using cached httplib2-0.31.2-py3-none-any.whl (91 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.5/324.5 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 15.8 MB/s eta 0:00:00m eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.2/173.2 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 6.6 MB/s eta 0:00:00
Using cached pyasn1_modules-0.4.2-py3-none-any.whl (181 kB

In [4]:
import ee

In [5]:
PROJECT_ID = "ee-shashigharti"  # <-- replace with YOUR project id

In [7]:
print("Earth Engine initialized successfully.")
!pip -q install geemap ipywidgets folium

Earth Engine initialized successfully.


In [8]:
import ee
import geemap
import ipywidgets as widgets
from IPython.display import display

In [ ]:
try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

In [ ]:
print("Earth Engine initialized with project:", PROJECT_ID)

In [ ]:
# ----------------------------
# 1) Basemaps: Google Hybrid (fallback to Esri)
# ----------------------------
GOOGLE_HYBRID_URL = "https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}"

In [ ]:
def add_google_hybrid(m):
    try:
        m.add_tile_layer(url=GOOGLE_HYBRID_URL, name="Google Hybrid", attribution="Google")
    except Exception:
        pass

In [ ]:
def add_esri_fallback(m):
    m.add_basemap("Esri.WorldImagery")
    m.add_basemap("Esri.WorldTransportation")

In [ ]:
# ----------------------------
# 2) EMIT helper: nearest band for wavelength (nm)
# ----------------------------
def nearest_emit_band_name(img, target_nm: float) -> ee.String:
    wl = ee.List(img.get("reflectance_wavelengths"))
    diffs = wl.map(lambda x: ee.Number(x).subtract(target_nm).abs())
    idx = diffs.indexOf(ee.Number(diffs.reduce(ee.Reducer.min())))
    return ee.String("reflectance_").cat(ee.Number(idx).format())

In [ ]:
# ----------------------------
# 3) AOI presets (coffee and cocoa)
#   Kenya coffee: Central Highlands near Nyeri
#   Ghana cocoa: Ashanti / Western region style belt (example near Kumasi)
#   Circles sized to ~3.1 km2 (radius ~1000 m)
# ----------------------------
def circle_aoi(lon, lat, radius_m=1000):
    return ee.Geometry.Point([lon, lat]).buffer(radius_m).bounds()

In [ ]:
AOI_COFFEE_KENYA = circle_aoi(36.95, -0.42, 1000)   # Nyeri coffee zone
AOI_COCOA_GHANA  = circle_aoi(-1.62,  6.69, 1000)   # Kumasi cocoa belt

In [ ]:
# ----------------------------
# 4) Global state
# ----------------------------
state = {
    "aoi": None,
    "emit_img": None,
    "s2_img": None,
    "training_fc": ee.FeatureCollection([]),
    "class_img": None,     # 0 other, 1 coffee, 2 cocoa
    "ndvi": None,
    "ndre": None,
    "plots_fc": None,      # plot polygons with crop + health
    "last_plot_layer": None,
    "last_crop_layer": None
}

In [ ]:
drawn_geoms = []

In [ ]:
# ----------------------------
# 5) Build interactive map
#   Default center on Kenya coffee belt
# ----------------------------
m = geemap.Map(center=[-0.42, 36.95], zoom=12)
add_google_hybrid(m)
add_esri_fallback(m)
m.addLayerControl()

In [ ]:
m.add_draw_control()
m.draw_control.clear()

In [ ]:
def handle_draw(target, action, geo_json):
    if action == "created":
        drawn_geoms.append(ee.Geometry(geo_json["geometry"]))
        print("Geometry captured.")
    elif action == "deleted":
        drawn_geoms.clear()
        print("All drawings cleared.")

In [ ]:
m.draw_control.on_draw(handle_draw)

In [ ]:
# ----------------------------
# 6) UI widgets
# ----------------------------
status = widgets.HTML("<b>Status:</b> Use preset AOI or draw your own (2–5 km²).")

In [ ]:
btn_set_aoi_drawn = widgets.Button(description="Set AOI (last drawn)", button_style="primary")
btn_set_aoi_coffee = widgets.Button(description="Set AOI Coffee Kenya", button_style="success")
btn_set_aoi_cocoa  = widgets.Button(description="Set AOI Cocoa Ghana",  button_style="success")

In [ ]:
btn_load = widgets.Button(description="Load EMIT + S2", button_style="warning")

In [ ]:
btn_add_coffee = widgets.Button(description="Add last drawn as COFFEE", button_style="success")
btn_add_cocoa  = widgets.Button(description="Add last drawn as COCOA", button_style="success")
btn_add_other  = widgets.Button(description="Add last drawn as OTHER", button_style="info")

In [ ]:
btn_train = widgets.Button(description="Train classifier + Crop map", button_style="danger")
btn_make_plots = widgets.Button(description="Build plot polygons + Health", button_style="danger")

In [ ]:
btn_export = widgets.Button(description="Export (Crop map + Plots)", button_style="")
btn_clear = widgets.Button(description="Clear all", button_style="")

In [ ]:
crop_filter = widgets.SelectMultiple(
    options=[("Coffee", "coffee"), ("Cocoa", "cocoa")],
    value=("coffee", "cocoa"),
    description="Crop",
    disabled=False
)

In [ ]:
health_filter = widgets.SelectMultiple(
    options=[("Healthy", 2), ("Moderate", 1), ("Stressed", 0)],
    value=(2, 1, 0),
    description="Health",
    disabled=False
)

In [ ]:
btn_apply_filters = widgets.Button(description="Apply filters (show plots)", button_style="primary")

In [ ]:
ui = widgets.VBox([
    status,
    widgets.HBox([btn_set_aoi_drawn, btn_set_aoi_coffee, btn_set_aoi_cocoa, btn_load, btn_clear]),
    widgets.HBox([btn_add_coffee, btn_add_cocoa, btn_add_other]),
    widgets.HBox([btn_train, btn_make_plots, btn_export]),
    widgets.HBox([crop_filter, health_filter, btn_apply_filters]),
])

In [ ]:
display(ui)
display(m)

In [ ]:
def set_status(msg):
    status.value = f"<b>Status:</b> {msg}"

In [ ]:
def last_geom():
    return drawn_geoms[-1] if len(drawn_geoms) else None

In [ ]:
def remove_layer_if_exists(layer_obj):
    if layer_obj is None:
        return
    try:
        m.remove_layer(layer_obj)
    except Exception:
        pass

In [ ]:
def set_aoi_geom(g, label):
    area_km2 = g.area(1).divide(1e6).getInfo()
    state["aoi"] = g
    m.addLayer(g, {"color": "black"}, f"AOI: {label}", True)
    set_status(f"AOI set to {label}. area={area_km2:.3f} km². Now click Load EMIT + S2.")

In [ ]:
# ----------------------------
# AOI buttons
# ----------------------------
def on_set_aoi_drawn(_):
    g = last_geom()
    if g is None:
        set_status("No polygon found. Draw AOI and double click to finish.")
        return
    set_aoi_geom(g, "Drawn")
btn_set_aoi_drawn.on_click(on_set_aoi_drawn)

In [ ]:
def on_set_aoi_coffee(_):
    set_aoi_geom(AOI_COFFEE_KENYA, "Coffee Kenya")
    m.set_center(36.95, -0.42, 12)
btn_set_aoi_coffee.on_click(on_set_aoi_coffee)

In [ ]:
def on_set_aoi_cocoa(_):
    set_aoi_geom(AOI_COCOA_GHANA, "Cocoa Ghana")
    m.set_center(-1.62, 6.69, 12)
btn_set_aoi_cocoa.on_click(on_set_aoi_cocoa)

In [ ]:
# ----------------------------
# 7) Load EMIT + S2 and show NDVI NDRE layers
# ----------------------------
def on_load(_):
    if state["aoi"] is None:
        set_status("Set AOI first.")
        return
    aoi = state["aoi"]

    emit_ic = (ee.ImageCollection("NASA/EMIT/L2A/RFL")
               .filterBounds(aoi)
               .filterDate("2022-01-01", "2026-01-01")
               .sort("system:time_start", False))

    n_emit = emit_ic.size().getInfo()
    if n_emit == 0:
        set_status("No EMIT scenes intersect this AOI. Try the other preset, or move AOI to intersect EMIT swath.")
        return
    emit = ee.Image(emit_ic.first())
    state["emit_img"] = emit

    def mask_s2(img):
        qa = img.select("QA60")
        cloud = qa.bitwiseAnd(1 << 10).eq(0)
        cirrus = qa.bitwiseAnd(1 << 11).eq(0)
        return img.updateMask(cloud.And(cirrus)).divide(10000)

    s2_ic = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
             .filterBounds(aoi)
             .filterDate("2023-01-01", "2025-01-01")
             .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
             .map(mask_s2))

    if s2_ic.size().getInfo() == 0:
        set_status("No Sentinel-2 scenes found. Widen date range or relax cloud filter.")
        return
    s2 = ee.Image(s2_ic.median())
    state["s2_img"] = s2

    ndvi = s2.normalizedDifference(["B8", "B4"]).rename("NDVI").clip(aoi)
    state["ndvi"] = ndvi

    b705 = nearest_emit_band_name(emit, 705.0)
    b790 = nearest_emit_band_name(emit, 790.0)
    ndre = emit.select(b790).subtract(emit.select(b705)).divide(emit.select(b790).add(emit.select(b705))).rename("NDRE").clip(aoi)
    state["ndre"] = ndre

    # RGB context
    b_r = nearest_emit_band_name(emit, 650.0)
    b_g = nearest_emit_band_name(emit, 560.0)
    b_b = nearest_emit_band_name(emit, 470.0)
    m.addLayer(emit.select([b_r, b_g, b_b]).clip(aoi), {"min": 0.0, "max": 0.3}, "EMIT RGB", False)
    m.addLayer(s2.select(["B4","B3","B2"]).clip(aoi), {"min": 0.02, "max": 0.35}, "S2 RGB", False)

    m.addLayer(ndvi, {"min": 0.0, "max": 0.9, "palette": ["d7191c", "ffffbf", "1a9641"]}, "NDVI (S2)", True)
    m.addLayer(ndre, {"min": -0.2, "max": 0.4, "palette": ["2c7bb6", "ffffbf", "d7191c"]}, "NDRE (EMIT)", True)

    set_status(f"Loaded EMIT scenes={n_emit}. Now add training polygons: coffee, cocoa, other.")
btn_load.on_click(on_load)

In [ ]:
# ----------------------------
# 8) Training polygons (auto buffer)
# ----------------------------
def add_training(lbl, name, buffer_m=150):
    g = last_geom()
    if g is None:
        set_status("Draw a training polygon first (small is fine).")
        return
    g_buf = g.buffer(buffer_m)
    feat = ee.Feature(g_buf, {"class": int(lbl), "class_name": name})
    state["training_fc"] = state["training_fc"].merge(ee.FeatureCollection([feat]))
    set_status(f"Added training: {name}. Total polygons={state['training_fc'].size().getInfo()}.")

In [ ]:
btn_add_coffee.on_click(lambda _: add_training(1, "coffee"))
btn_add_cocoa.on_click(lambda _: add_training(2, "cocoa"))
btn_add_other.on_click(lambda _: add_training(0, "other"))

In [ ]:
# ----------------------------
# 9) Train classifier
# ----------------------------
def on_train(_):
    if state["emit_img"] is None or state["aoi"] is None:
        set_status("Load EMIT first.")
        return

    aoi = state["aoi"]
    emit = state["emit_img"]

    targets = [450, 500, 550, 600, 650, 680, 705, 720, 740, 760, 790, 820, 860, 900, 970, 1100, 1200]
    bands = [nearest_emit_band_name(emit, t) for t in targets]
    emit_sel = emit.select(bands).clip(aoi)

    training = emit_sel.sampleRegions(state["training_fc"], ["class"], scale=30, geometries=True)
    if training.size().getInfo() < 50:
        training = emit_sel.sampleRegions(state["training_fc"], ["class"], scale=60, geometries=True)

    clf = ee.Classifier.smileRandomForest(300, 2).train(training, "class", emit_sel.bandNames())
    class_img = emit_sel.classify(clf).rename("crop_class").clip(aoi)
    state["class_img"] = class_img

    remove_layer_if_exists(state["last_crop_layer"])
    crop_vis = {"min": 0, "max": 2, "palette": ["808080", "00ff00", "ff00ff"]}
    state["last_crop_layer"] = m.addLayer(class_img, crop_vis, "Crop map (other/coffee/cocoa)", True)

    set_status("Crop map created. Next: Build plot polygons + health.")
btn_train.on_click(on_train)

In [ ]:
# ----------------------------
# 10) Plot polygons + health
# ----------------------------
def build_plots_for_crop(mask, crop_name, aoi, scale_m=60):
    labeled = mask.selfMask().connectedComponents(ee.Kernel.square(1), 1024).select("labels").rename("plot_id")
    plots = labeled.reduceToVectors(aoi, scale_m, "polygon", "plot_id", True, 1e13)
    return plots.map(lambda f: f.set({"crop": crop_name}))

In [ ]:
def on_make_plots(_):
    if any(state[k] is None for k in ["aoi","class_img","ndvi","ndre"]):
        set_status("Need crop map + NDVI + NDRE first.")
        return

    aoi = state["aoi"]
    crop_img = state["class_img"]
    ndvi = state["ndvi"]
    ndre = state["ndre"]

    coffee_plots = build_plots_for_crop(crop_img.eq(1), "coffee", aoi, 60)
    cocoa_plots  = build_plots_for_crop(crop_img.eq(2), "cocoa",  aoi, 60)
    plots = coffee_plots.merge(cocoa_plots)

    plots = plots.map(lambda f: f.set({
        "ndvi_mean": ndvi.reduceRegion(ee.Reducer.mean(), f.geometry(), 20, maxPixels=1e9).get("NDVI"),
        "ndre_mean": ndre.reduceRegion(ee.Reducer.mean(), f.geometry(), 60, maxPixels=1e9).get("NDRE")
    }))

    def add_health(f):
        nv = ee.Number(f.get("ndvi_mean"))
        nr = ee.Number(f.get("ndre_mean"))
        healthy  = nv.gte(0.60).And(nr.gte(0.20))
        stressed = nv.lt(0.45).Or(nr.lt(0.10))
        moderate = healthy.Not().And(stressed.Not())
        h = ee.Number(0).where(moderate, 1).where(healthy, 2)
        return f.set({"health": h, "area_ha": f.geometry().area(1).divide(10000)})

    state["plots_fc"] = plots.map(add_health)
    set_status("Plots created and health labeled. Now Apply filters to display.")
btn_make_plots.on_click(on_make_plots)

In [ ]:
# ----------------------------
# 11) Apply filters and display plots
# ----------------------------
def style_plots(fc):
    stressed = fc.filter(ee.Filter.eq("health", 0)).map(lambda f: f.set({"style": {"color":"#d7191c","fillColor":"#d7191c","width":2,"fillOpacity":0.35}}))
    moderate = fc.filter(ee.Filter.eq("health", 1)).map(lambda f: f.set({"style": {"color":"#fdae61","fillColor":"#fdae61","width":2,"fillOpacity":0.35}}))
    healthy  = fc.filter(ee.Filter.eq("health", 2)).map(lambda f: f.set({"style": {"color":"#1a9641","fillColor":"#1a9641","width":2,"fillOpacity":0.35}}))
    return stressed.merge(moderate).merge(healthy)

In [ ]:
def on_apply_filters(_):
    if state["plots_fc"] is None:
        set_status("Build plot polygons first.")
        return

    crops = list(crop_filter.value)
    healths = list(health_filter.value)

    fc = state["plots_fc"]

    crop_filters = []
    if "coffee" in crops: crop_filters.append(ee.Filter.eq("crop", "coffee"))
    if "cocoa"  in crops: crop_filters.append(ee.Filter.eq("crop", "cocoa"))
    crop_f = crop_filters[0] if len(crop_filters)==1 else ee.Filter.Or(*crop_filters)
    fc = fc.filter(crop_f)

    health_filters = [ee.Filter.eq("health", int(h)) for h in healths]
    hf = health_filters[0] if len(health_filters)==1 else ee.Filter.Or(*health_filters)
    fc = fc.filter(hf)

    fc_styled = style_plots(fc)

    remove_layer_if_exists(state["last_plot_layer"])
    state["last_plot_layer"] = m.addLayer(fc_styled.style(styleProperty="style"), {}, "Plots (filtered) health", True)

    set_status("Filtered plots displayed. Toggle NDVI and NDRE for comparison.")
btn_apply_filters.on_click(on_apply_filters)

In [ ]:
# ----------------------------
# 12) Export outputs
# ----------------------------
def on_export(_):
    if any(state[k] is None for k in ["aoi","class_img","plots_fc","ndvi","ndre"]):
        set_status("Need crop map + plots + indices first.")
        return
    aoi = state["aoi"]
    geemap.ee_export_image_to_drive(state["class_img"], "crop_map_other_coffee_cocoa", "GEE_Exports", aoi, 60, 1e13)
    geemap.ee_export_image_to_drive(state["ndvi"], "ndvi_sentinel2", "GEE_Exports", aoi, 20, 1e13)
    geemap.ee_export_image_to_drive(state["ndre"], "ndre_emit_hyperspectral", "GEE_Exports", aoi, 60, 1e13)
    geemap.ee_export_vector_to_drive(state["plots_fc"], "coffee_cocoa_plots_with_health", "GEE_Exports", file_format="SHP")
    set_status("Export tasks created. Run them in Earth Engine Tasks.")
btn_export.on_click(on_export)

In [ ]:
# ----------------------------
# 13) Clear all
# ----------------------------
def on_clear(_):
    drawn_geoms.clear()
    m.draw_control.clear()
    for k in state.keys():
        state[k] = None if k != "training_fc" else ee.FeatureCollection([])
    set_status("Cleared everything. Start again with Coffee Kenya or Cocoa Ghana preset AOI.")
btn_clear.on_click(on_clear)